# Experiment !!! :D

In [43]:
# IMPORTS
import sys
import os
import time

from point import *
from dataset import *

# """ Runtime parameters """
# assignment_nr = 1       # The assignment number. Used by the visualizer to determine what has to be visualized

In [44]:
# HELPER FUNCTIONS

def compute_centroids(cluster_points, n, d, k):
    """
    Computes the optimal set of centroids for a fixed assignment of cluster labels to the input points

    :param cluster_points: cluster points of ClusterPoint class
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """

    cluster_sizes = [0] * k # initialize cluster sizes
    point_sum = [ClusterPoint(dimension=d) for _ in range(k)] # initalize point sums
    centroids = [CentroidPoint(dimension=d) for _ in range(k)] # initialize centroids

    for i in range(n):
        cluster_sizes[cluster_points[i].cluster_label] += 1
        point_sum[cluster_points[i].cluster_label].add(other=cluster_points[i])
    print('computed cluster sizes', cluster_sizes)
    print('computed point sum', [i.coords for i in point_sum])

    for i in range(k):
        point_sum[i].div(cluster_sizes[i])
        centroids[i] = point_sum[i]
    print('computed centroids', [i.coords for i in centroids], '\n')

    return centroids


def compute_labels(cluster_points, centroids, n, d, k):
    """
    Assigns cluster points to centroids.

    :param cluster_points: cluster pointsn of ClusterPoint class
    :param centroids: previously defined centroids
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """

    for i in range(n):
        min_distance = 999999 # initialize high min distance
        for j in range(k):
            current_distance = cluster_points[i].sq_distance_to(other=centroids[j])
            if current_distance < min_distance:
                min_distance = current_distance
                cluster_label = j
        cluster_points[i].cluster_label = cluster_label

    return cluster_points

def read(path_in):
    """
    Reads the input set, clusters the points and writes to output

    :param path_in:     location of the input set
    :param path_out:    location to print the output
    """

    # read input from file
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
            return input_obj
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return None

def uniform_random():
    """
    Find x uniformly at random such that 0 < x < 1.
    """
    none = True

    while none:
        x = random.random()

        if 0 < x < 1:
            return x

In [45]:
import random
import math

In [46]:
# INITIALIZE CENTROIDS

def initialize_centroids(input_obj, method="first_k"):
    """
    Calculates the initial centroid placement

    :param input_obj:   the input object
    :param method: initialization method, possible options are first_k, gonzales, kmeans++
    :return:            a list of k centroids in the plane
    """
    if method=="first_k":
        ### IMPLEMENTATION: FIRST K CLUSTER POINTS AS INITIAL CENTROIDS
        centroids = [CentroidPoint(p.dimension, list(p.coords)) for p in input_obj.cluster_points[:input_obj.k]]

    if method=="gonzales":
        ### IMPLEMENTATION: GONZALES
        cluster_points = input_obj.cluster_points
        centroids = [CentroidPoint(dimension=input_obj.d) for _ in range(input_obj.k)] # initialize centroids # TODO not needed
        min_distances = [0] * input_obj.n

        # initialize first centroids with first input point
        centroids[0] = cluster_points[0]

        for j in range(input_obj.n):
            min_distances[j] = 999999 # initialize high min distances

        for i in range(1, input_obj.k):
            # update distance to closest centroid for each input point
            for j in range(input_obj.n):
                x = cluster_points[j].sq_distance_to(other=centroids[i-1])
                if min_distances[j] > x:
                    min_distances[j] = x

            # compute input point which is farthest from current set of centroids
            max_distance = 0 # initialize lowest max distance

            for j in range(input_obj.n):
                if max_distance < min_distances[j]:
                    max_distance = min_distances[j]
                    farthest_point = cluster_points[j]

            # add farthest point to set of centroids
            centroids[i] = farthest_point

    if method=="kmeans++":
        ### IMPLEMENTATION: KMEANS++
        points, n, d, k = input_obj.cluster_points, input_obj.n, input_obj.d, input_obj.k
        min_distances = [0] * n
        cumulative = [0] * n

        centroids = [None] * k # initialise before indexing

        x = uniform_random()
        sampled_index = math.ceil(x * n)
        centroids[0] = points[sampled_index]

        for j in range(n):
            min_distances[j] = float('inf')

        for i in range(1, k):
            for j in range(n):
                x = points[j].sq_distance_to(centroids[i-1])

                if min_distances[j] > x:
                    min_distances[j] = x

            cumulative[0] = min_distances[0] * min_distances[0]
            for j in range(1, n):
                cumulative[j] = cumulative[j-1] + (min_distances[j] * min_distances[j])

            x = uniform_random()
            x = x * cumulative[n-1]

            if x <= cumulative[0]:
                sampled_index = 0
            else:
                for j in range(1, n):
                    if (x > cumulative[j-1]) and (x <= cumulative[j]):
                        sampled_index = j

            centroids[i] = points[sampled_index]

    return centroids

In [47]:
# PERFORM CLUSTERING

def cluster(input_obj, method="first_k"):
    """
    Perform k-means clustering on the input set

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """

    # initalize centroids
    centroids_old = initialize_centroids(input_obj, method=method)

    # obtain n, d, k once
    # initialize cluster points
    n = input_obj.n
    d = input_obj.d
    k = input_obj.k
    cluster_points = input_obj.cluster_points

    # repeat until centroids did not change in the last iteration
    i = 0
    while True:
        print('iteration', i)

        cluster_points = compute_labels(cluster_points, centroids_old, n, d, k)
        centroids_new = compute_centroids(cluster_points, n, d, k)

        # stop if unchanged
        if centroids_new == centroids_old:
            break

        centroids_old = centroids_new

        i+=1

    centroids = centroids_new

    return centroids, i

In [48]:
# RUN AND WRITE TO OUTPUT

def run(path_in, path_out, method):
    """
    Reads the input dataset, runs k-means, writes output, and returns metrics.

    :param path_in:   location of the input file
    :param path_out:  location to write the output file
    :param method:    initialisation method ('gonzales' or 'kmeans++')
    :return:          (sse, n_iterations, wall_clock_seconds)
                        sse              – final sum of squared errors (inertia)
                        n_iterations     – Lloyd iterations until convergence
                        wall_clock_secs  – total wall-clock runtime in seconds
    """
    # ── Read input ────────────────────────────────────────────────────────────
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return None, None, None

    # ── Run clustering (timed) ────────────────────────────────────────────────
    t_start = time.perf_counter()                      # high-resolution timer
    centroids, n_iterations = cluster(input_obj, method=method)
    t_end   = time.perf_counter()

    wall_clock_secs = t_end - t_start

    # ── Compute metrics ───────────────────────────────────────────────────────
    assert len(centroids) == input_obj.k

    sse = input_obj.score(centroids)                   # sum of squared errors

    # ── Write output file ─────────────────────────────────────────────────────
    try:
        input_obj.write_output(centroids, path_out, 1)
    except IOError:
        print("Could not write output to file: " + path_out, file=sys.stderr)

    # ── Diagnostic stderr ─────────────────────────────────────────────────────
    print(f"[{method}] k={input_obj.k}  "
          f"SSE={sse:.3f}  "
          f"iters={n_iterations}  "
          f"time={wall_clock_secs:.3f}s",
          file=sys.stderr)

    return sse, n_iterations, wall_clock_secs

In [49]:
# RUN EXPERIMENT

# Runs gonzales and kmeans++ for every (k, r) pair.
# Writes per-run output files to output/ and a combined summary to results/.

import os
import sys
from collections import defaultdict

# Experiment parameters
k_values = [2, 5, 10, 20, 50, 100]
R        = 10                                  # runs per k (seeds 1, ..., 10)
methods  = ["gonzales", "kmeans++"]

INPUT_DIR   = "input"
OUTPUT_DIR  = "output"
RESULTS_DIR = "results"
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Storage: results[(method, k)] = list of (sse, iters, time)
results = defaultdict(list)

total_runs = len(methods) * len(k_values) * R
run_nr     = 0

for method in methods:
    for k in k_values:
        for r in range(1, R + 1):
            run_nr += 1
            path_in  = os.path.join(INPUT_DIR,  f"k{k}_R{r}.in")
            path_out = os.path.join(OUTPUT_DIR, f"{method}_k{k}_R{r}.out")

            print(f"[{run_nr}/{total_runs}] method={method}  k={k}  R={r}",
                  file=sys.stderr)

            sse, iters, t = run(path_in, path_out, method)

            if sse is not None:            # guard against read errors
                results[(method, k)].append((sse, iters, t))

# Compute averages and write summary
summary_path = os.path.join(RESULTS_DIR, "summary.txt")

col_w = 14   # column width for alignment

header = (
    f"{'Method':<12} "
    f"{'k':>{col_w}} "
    f"{'SSE_mean':>{col_w}} "
    f"{'SSE_std':>{col_w}} "
    f"{'Iters_mean':>{col_w}} "
    f"{'Iters_std':>{col_w}} "
    f"{'Time_mean(s)':>{col_w}} "
    f"{'Time_std(s)':>{col_w}}"
)
separator = "-" * len(header)

lines = [
    "K-Means Initialization Experiment — Summary",
    f"Runs per (method, k): {R}  |  n=10,000  |  d=2  |  cluster_std=0.3",
    separator,
    header,
    separator,
]

for method in methods:
    for k in k_values:
        runs = results[(method, k)]
        if not runs:
            continue

        sses, iters_list, times = zip(*runs)

        sse_mean   = sum(sses)        / len(sses)
        sse_std    = (sum((x - sse_mean)**2   for x in sses)        / len(sses)) ** 0.5
        iter_mean  = sum(iters_list)  / len(iters_list)
        iter_std   = (sum((x - iter_mean)**2  for x in iters_list)  / len(iters_list)) ** 0.5
        time_mean  = sum(times)       / len(times)
        time_std   = (sum((x - time_mean)**2  for x in times)       / len(times)) ** 0.5

        lines.append(
            f"{method:<12} "
            f"{k:{col_w}d} "
            f"{sse_mean:{col_w}.2f} "
            f"{sse_std:{col_w}.2f} "
            f"{iter_mean:{col_w}.2f} "
            f"{iter_std:{col_w}.2f} "
            f"{time_mean:{col_w}.4f} "
            f"{time_std:{col_w}.4f}"
        )
    lines.append("")   # blank line between methods

lines.append(separator)

with open(summary_path, "w") as f:
    f.write("\n".join(lines) + "\n")

print(f"\nSummary written to {summary_path}")

# Also print to notebook
print("\n".join(lines))

[1/120] method=gonzales  k=2  R=1
[gonzales] k=2  SSE=1799.906  iters=1  time=0.067s
[2/120] method=gonzales  k=2  R=2


iteration 0
computed cluster sizes [5000, 5000]
computed point sum [[-8275.571268751577, 22039.26698541081], [-49958.928014087585, -19768.730306428937]]
computed centroids [[-1.6551142537503154, 4.407853397082162], [-9.991785602817517, -3.9537460612857873]] 

iteration 1
computed cluster sizes [5000, 5000]
computed point sum [[-8275.571268751577, 22039.26698541081], [-49958.928014087585, -19768.730306428937]]
computed centroids [[-1.6551142537503154, 4.407853397082162], [-9.991785602817517, -3.9537460612857873]] 

iteration 0
computed cluster sizes [5000, 5000]
computed point sum [[-6406.196104424254, -47459.569891852116], [4965.240725136727, -6412.598322225789]]
computed centroids [[-1.2812392208848509, -9.491913978370423], [0.9930481450273454, -1.2825196644451577]] 

iteration 1
computed cluster sizes [5000, 5000]
computed point sum [[-6406.196104424254, -47459.569891852116], [4965.240725136727, -6412.598322225789]]
computed centroids [[-1.2812392208848509, -9.491913978370423], [0.99

[gonzales] k=2  SSE=1813.352  iters=1  time=0.051s
[3/120] method=gonzales  k=2  R=3
[gonzales] k=2  SSE=1782.158  iters=1  time=0.050s
[4/120] method=gonzales  k=2  R=4


computed cluster sizes [5000, 5000]
computed point sum [[5036.265202834578, 20773.706381732332], [-20892.999108555643, 1072.7958246468938]]
computed centroids [[1.0072530405669156, 4.154741276346466], [-4.178599821711129, 0.21455916492937877]] 

iteration 1
computed cluster sizes [5000, 5000]
computed point sum [[5036.265202834578, 20773.706381732332], [-20892.999108555643, 1072.7958246468938]]
computed centroids [[1.0072530405669156, 4.154741276346466], [-4.178599821711129, 0.21455916492937877]] 

iteration 0
computed cluster sizes [5004, 4996]
computed point sum [[46738.089140437565, 4744.469837815522], [47241.12064381298, 21456.774543532018]]
computed centroids [[9.340145711518298, 0.9481354591957477], [9.455788759770412, 4.294790741299443]] 

iteration 1
computed cluster sizes [5000, 5000]
computed point sum [[46700.03513975083, 4731.239549044604], [47279.17464449971, 21470.004832302937]]
computed centroids [[9.340007027950165, 0.9462479098089208], [9.455834928899943, 4.29400096646

[gonzales] k=2  SSE=1780.564  iters=2  time=0.073s
[5/120] method=gonzales  k=2  R=5


...

[kmeans++] k=100  SSE=1689.729  iters=33  time=26.895s
[119/120] method=kmeans++  k=100  R=9


iteration 0
computed cluster sizes [106, 100, 36, 196, 89, 48, 75, 100, 112, 180, 153, 87, 113, 107, 14, 148, 97, 68, 98, 134, 112, 212, 102, 100, 193, 45, 96, 99, 125, 71, 99, 101, 96, 136, 62, 200, 114, 218, 104, 59, 88, 145, 195, 65, 119, 95, 102, 11, 26, 70, 71, 86, 120, 113, 98, 105, 82, 175, 94, 56, 87, 71, 101, 77, 151, 93, 67, 61, 95, 144, 92, 21, 35, 100, 89, 140, 146, 68, 109, 99, 60, 137, 115, 93, 105, 187, 69, 32, 110, 54, 80, 79, 100, 100, 143, 92, 87, 67, 61, 62]
computed point sum [[-22.49479838980962, 431.7083087069954], [979.3952849661745, -655.3717007616506], [-270.43841457134965, -212.89393140645782], [-42.036967226848994, -241.56811116473935], [599.1274040899234, 49.76551165987607], [171.80068724324389, 144.86143628289744], [239.1981886979085, -334.20017816881335], [-395.328904497135, 937.1775171293148], [986.8792305150373, 678.2115312985675], [-1601.415123013364, 1017.6471086923464], [-286.8280288191421, -1113.2243158943006], [391.87542485458886, 701.3566055740911]

[kmeans++] k=100  SSE=1681.197  iters=14  time=12.340s
[120/120] method=kmeans++  k=100  R=10


iteration 0
computed cluster sizes [136, 53, 23, 122, 101, 111, 99, 94, 129, 189, 178, 112, 100, 113, 158, 100, 180, 100, 100, 165, 97, 180, 93, 104, 167, 10, 20, 126, 135, 52, 178, 25, 74, 108, 135, 100, 98, 172, 97, 100, 46, 103, 196, 101, 65, 103, 173, 107, 100, 11, 157, 99, 118, 100, 78, 99, 77, 105, 107, 95, 81, 28, 93, 96, 80, 96, 43, 203, 67, 29, 223, 117, 78, 88, 94, 87, 98, 87, 85, 76, 62, 34, 86, 112, 236, 90, 74, 45, 91, 78, 97, 31, 119, 32, 64, 145, 64, 57, 101, 59]
computed point sum [[185.03880901393, 228.17993170097964], [-339.8293433140071, -458.48402491117156], [81.04848372782811, -144.02907267826254], [-1166.0651527212096, 1171.440099435722], [-996.542578690299, 28.755124869781294], [-250.55905491165, -744.7513485992453], [-536.5884080657906, -240.61336337334475], [577.1611642586579, 209.3975301349961], [248.12990136273635, 1029.6764382620863], [-194.44012159457193, -284.5409551251366], [-1633.7073902732234, -535.183413092681], [-290.7788420468502, 383.18060026697975]

[kmeans++] k=100  SSE=1679.538  iters=34  time=29.384s


In [50]:
run("input/k2_R1.in", "output/test.out", "kmeans++")

iteration 0
computed cluster sizes [5000, 5000]
computed point sum [[-49958.928014087585, -19768.730306428937], [-8275.571268751577, 22039.26698541081]]
computed centroids [[-9.991785602817517, -3.9537460612857873], [-1.6551142537503154, 4.407853397082162]] 

iteration 1
computed cluster sizes [5000, 5000]
computed point sum [[-49958.928014087585, -19768.730306428937], [-8275.571268751577, 22039.26698541081]]
computed centroids [[-9.991785602817517, -3.9537460612857873], [-1.6551142537503154, 4.407853397082162]] 



[kmeans++] k=2  SSE=1799.906  iters=1  time=0.053s


(1799.9062965507676, 1, 0.05328960000042571)